# AI 402 Module 5 — Lab 2: a mini evaluation harness
Runs top-to-bottom on a free Colab in seconds. No sign-in, no Drive, no installs, no file paths to edit:
all data is built into the cells. Run the cells in order, from the check below to B5.

In [ ]:
# @title ✅ Check the session (no installs needed)
import sys, platform

IN_COLAB = "google.colab" in sys.modules
print(f"Python {platform.python_version()}  |  Colab: {IN_COLAB}")
print("This notebook needs only the standard library.")

## The harness
Your Lab 1 page classifies memos into HR, Finance, Procurement or Records. The harness runs a
**fixed set of 10 memos with expected answers** through a `classify()` function, records latency and
pass / fail, and compares one run against the next — the regression tests of an AI system.

A simple keyword `classify()` is the model under test, so the notebook runs as-is. Any function with the
same shape (one text in, one label out) can take its place. Your Lab 1 page runs in the browser, so to
compare, paste the same 10 memos into it and note where the two disagree.

In [ ]:
# @title B1 · The evaluation set — 10 synthetic memos, expected answers fixed
# All text is synthetic. The set stays fixed, so scores from different runs mean something.
EVAL_SET = [
    {"id": 1,  "expected": "HR",          "text": "All employees are reminded to submit their Individual Development Plans for 2027 training by January 15."},
    {"id": 2,  "expected": "Finance",     "text": "Prepare the year-end financial reports and the liquidation of cash advances by December 20."},
    {"id": 3,  "expected": "Procurement", "text": "Invitation to Bid for the supply and delivery of twenty-five (25) laptop units for the regional offices."},
    {"id": 4,  "expected": "Procurement", "text": "Please forward the signed abstract of canvass and the supplier files to the BAC Secretariat; the records of the 2026 canvass are needed for the audit."},
    {"id": 5,  "expected": "HR",          "text": "Paalala sa lahat ng personnel: bukas na ang enrollment para sa 2027 Wellness Program hanggang Biyernes."},
    {"id": 6,  "expected": "Finance",     "text": "Budget call for the first semester of 2027: submit your division's proposed expenditures by March 1."},
    {"id": 7,  "expected": "Procurement", "text": "Submission of quotations for the repair of two (2) air-conditioning units at the Administration Building."},
    {"id": 8,  "expected": "Procurement", "text": "The Bids and Awards Committee will open the sealed proposals of shortlisted suppliers on Thursday; see the posting on PhilGEPS."},
    {"id": 9,  "expected": "Finance",     "text": "Reimbursement of travel expenses for the regional convention: submit your receipts to the Accounting Section by Monday."},
    {"id": 10, "expected": "Records",     "text": "Year-end housekeeping: all divisions must archive their 2026 files to the central records room by December 15."},
]
print(f"{len(EVAL_SET)} memos loaded:", " · ".join(m["expected"] for m in EVAL_SET))

In [ ]:
# @title B2 · The model under test
# The default below is a keyword rule so this notebook runs as-is, and it is imperfect on purpose:
# leave it as-is and you will see 7 of 10 pass. Any classify(text) -> label can replace it.
LABELS = ["HR", "Finance", "Procurement", "Records"]
KEYWORDS = {
    "HR":          ["employee", "personnel", "training", "performance", "wellness"],
    "Finance":     ["budget", "financial", "liquidation", "funds", "expenditure"],
    "Procurement": ["invitation to bid", "supply and delivery", "quotation", "purchase"],
    "Records":     ["records", "certified copies", "archive", "files", "registry"],
}

def classify(text: str) -> str:
    """Return the label with the most keyword hits; ties go to LABELS order; none is 'Unclassified'."""
    t = text.lower()
    scores = {lab: sum(t.count(kw) for kw in kws) for lab, kws in KEYWORDS.items()}
    best = max(scores.values())
    if best == 0:
        return "Unclassified"
    return next(lab for lab in LABELS if scores[lab] == best)

In [ ]:
# @title B3 · Run the harness: latency, output, pass / fail for every memo
import time

def run_eval(classifier=classify, label="run"):
    rows = []
    for m in EVAL_SET:
        t0 = time.perf_counter()
        pred = classifier(m["text"])
        rows.append({"id": m["id"], "expected": m["expected"], "predicted": pred,
                     "passed": pred == m["expected"], "seconds": time.perf_counter() - t0})
    print(f"{'#':>2}  {'expected':<11} {'predicted':<12} {'ms':>6}  result")
    print("-" * 52)
    for r in rows:
        mark = "PASS" if r["passed"] else "FAIL"
        print(f"{r['id']:>2}  {r['expected']:<11} {r['predicted']:<12} {r['seconds']*1000:>6.1f}  {mark}")
    print("-" * 52)
    print(f"{label}: {sum(r['passed'] for r in rows)} / {len(rows)} passed")
    return rows

results = run_eval()
baseline = {r["id"]: r["passed"] for r in results}

In [ ]:
# @title B4 · Change something, rerun, compare
# Edit classify() in cell B2 — add a keyword, change a rule, paste a new model — then run this cell.
# It reports what improved and what regressed. Nothing else moved, so every delta traces to your change.
results2 = run_eval()
after = {r["id"]: r["passed"] for r in results2}

changed = [m["id"] for m in EVAL_SET if baseline[m["id"]] != after[m["id"]]]
if not changed:
    print("\nNo change in outcomes — make the change bigger, or check that B2 was re-run.")
for m in EVAL_SET:
    if m["id"] in changed:
        print(f"  item {m['id']:>2}: {'improved' if after[m['id']] else 'REGRESSED'} — {m['text'][:60]}…")
print(f"\npass rate: {sum(after.values())}/10   (baseline was {sum(baseline.values())}/10)")

**Example change to try first:** the baseline misses two memos outright and misfiles one. Extending the
keyword lists fixes all three: add `"travel"` and `"expense"` to Finance, and `"canvass"`,
`"supplier"`, `"bids and awards"`, `"philgeps"` to Procurement — 10 of 10. Then try to break it
with a memo that mentions two divisions, or a division not in the list, and see what the harness catches.

**Who decides the right answer?** Memo 4 mentions records but is about a canvass, so this set calls it
Procurement. Your agency may disagree. Agreeing on the labels is part of building an evaluation set.
Memo 5 is in Taglish on purpose: real users write that way, so test for it.

In [ ]:
# @title B5 · Optional: Claude as judge, you as auditor
# Copies this transcript + rubric; paste it into Claude and compare its grading with the columns above.
latest = results2 if "results2" in globals() else results
transcript = "\n".join(
    f"{r['id']}. expected={r['expected']} predicted={r['predicted']} {'PASS' if r['passed'] else 'FAIL'}"
    for r in latest)
rubric = (
    "You are auditing an AI memo classifier. Grade this transcript item by item:\n"
    "1. Is the PASS/FAIL mark correct for each item?\n"
    "2. Which failures are keyword-rule problems, and which would still fail a smarter model?\n"
    "3. What memos should we ADD to catch weaknesses this set misses?\n"
    "Be specific. If you are unsure about an item, say so instead of guessing.")
print("Copy everything between the lines into Claude. Judges can be wrong too — spot-check its grades.")
print("-" * 60)
print(rubric, "\n\n", transcript, sep="")
print("-" * 60)

**What this taught:** version the evaluation set, run it before every change, and compare
scores before you release. Two runs differ only in what you changed — that is what makes the comparison
evidence rather than opinion.

*Back at your desk:* point the harness at your own system's classify(), and grow the set with real (anonymized)
questions — the failures are your new test cases.